# Power-scan campaign — analysis template

**What this is.** A general, copy-me notebook to analyse one *power scan*: a family of acquisitions where only the **pump power** changes. Duplicate this file per scan and edit the config cell.

**Pipeline (all logic lives in `src/`):**
1. `Campaign.from_powerscan(...)` discovers one merged run per power.
2. `GridVisualizer` lets you pick the g² integration window from the sweep plateau.
3. `Campaign.plot_overview()` shows g², R, singles and the normalised collapse vs power.
4. `PowerScanAnalyzer` fits the model $\langle I_n\rangle\sim I_0^{K(n)}$ and collapses $(g^{(2)}-1)/K^2\to\sigma^2$.
5. `CampaignReport(...).export()` writes a slide-ready folder under `results/`.

The model we are testing: classical pump noise gives $g_n^{(2)}=1+K(n)^2\sigma^2$, $g_{mn}^{(2)}=1+K(m)K(n)\sigma^2$, hence $R\le 1$ always.

In [ ]:
# --- bootstrap: make `src` importable and run from the project root ---
import sys, os
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # so 'data/...' and 'results/...' paths are consistent

from src import Campaign, PowerScanAnalyzer, GridVisualizer, CampaignReport, config
config.apply_style(usetex=True)  # set usetex=False on machines without a TeX install

## 1. Configure & load
Point `SCAN_DIR` at the power-scan folder (the one containing a `MERGED/` sub-folder).

In [ ]:
SCAN_DIR  = 'data/Jun16/PowerScan_GaAs100_P1'   # <-- edit me
HARMONICS = (3, 5)                              # harmonics present in this cut (GaAs100 has no H4)
TAU_INT   = 4.0                                 # g2 integration window (ns); refine in step 2

camp = Campaign.from_powerscan(SCAN_DIR, harmonics=HARMONICS, tau_in_ns=TAU_INT)
print(camp.runs[0].acquisition_summary())
camp.summary_table()

## 2. Choose the integration window $\tau_{in}$
Look for the plateau where g² stops depending on the window, and set `TAU_INT` above accordingly.

In [ ]:
gv = GridVisualizer(camp.runs, labels=camp.labels, comparison_variable='Pump power')
gv.plot_coherence(xlim=8.0, integration_window_ns=TAU_INT)
gv.plot_g2(methods=['delay'], tau_min=0.3, tau_max=25, step=1.0);

## 3. Campaign overview
g²(0), Cauchy-Schwarz R, harmonic singles and the normalised excess, all vs pump power.

In [ ]:
camp.plot_overview(g2_ylim=(0.9, 1.6), R_ylim=(0.9, 1.1));

## 4. Power-scan model: $K(n)$ scaling and the collapse to $\sigma^2$
`PowerScanAnalyzer` measures $K(n)=\mathrm{d}\ln\langle I_n\rangle/\mathrm{d}\ln I_0$ at each power and rescales every g² curve; they should collapse onto the same pump excess $g_0^{(2)}-1=\sigma^2$.

In [ ]:
psa = PowerScanAnalyzer(camp.runs, harmonics=HARMONICS, tau_in_ns=TAU_INT)
psa.summary_table()
psa.plot_overview(harmonics=HARMONICS);

## 5. Full detector grids (optional)
Per-detector g² and R vs power (TT/TR/RT/RR) for a closer look.

In [ ]:
gv.plot_power_scan_g2(tau_in_ns=TAU_INT)
gv.plot_power_scan_R(tau_in_ns=TAU_INT, ylim=(0.96, 1.04));

## 6. Export the report
Writes tables, CSVs and all figures (including the model) to `results/<scan name>/`.

In [ ]:
CampaignReport(camp, powerscan=psa).export(g2_ylim=(0.9, 1.6), R_ylim=(0.9, 1.1))